In [9]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
from itables import init_notebook_mode
from open_dataset_store import quick_start
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


store = quick_start('./ExperimentResults', backend='local')

init_notebook_mode(all_interactive=True)

# import itables.options as opt
# opt.lengthMenu = [10, 25, 50]
# opt.scrollX = True

CSV_PATH = './results/state_log.csv'

df = pd.read_csv(CSV_PATH)
df = df.copy()
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)
df['timestamp'] = df['Datetime'].astype('int64') // 10**9
ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')

# list(df.columns)
# df.head(n=20)
# sum = store.get_df_summary(df, detailed=True)

Store initialised at: ./ExperimentResults (Backend: local)


Loaded 480 timesteps, columns: 307


In [10]:
# Global Styles
SOURCE_STYLES = {
    "Zone":     {"color": "#1f77b4", "dash": "solid",   "width": 2},    
    "Outside":  {"color": "#ff7f0e", "dash": "dash",    "width": 0.5},  
    "Supply":   {"color": "#2ca02c", "dash": "dash",    "width": 0.5},  
    "Other_1":  {"color": "#FFBE91", "dash": "solid",   "width": 2},    
    "Setpoint": {"color": "#CFEBFF", "dash": "dot",     "width": 0.5},  
}

def build_zone_subplots(df, zone_name, subplot_config, source_styles=SOURCE_STYLES, row_height=250):
    total_rows = len(subplot_config)
    
    fig = make_subplots(
        rows=total_rows,
        cols=1,
        shared_xaxes=False,
        subplot_titles=[panel["title"] for panel in subplot_config],
        specs=[[{"secondary_y": True}]] * total_rows
    )
    
    for row_idx, panel in enumerate(subplot_config, start=1):
        
        # --- 1. Add Data Traces ---
        for trace_info in panel["traces"]:
            col_name = trace_info["col"]
            source_type = trace_info["source"]
            
            if col_name in df.columns:
                base_style = source_styles.get(source_type, {"color": "black", "dash": "solid", "width": 1})
                line_color = trace_info.get("color", base_style["color"])
                line_dash = trace_info.get("dash", base_style["dash"])
                line_width = trace_info.get("width", base_style["width"])
                
                display_name = trace_info.get("name", col_name)
                is_secondary = trace_info.get("secondary_y", False)
                
                fig.add_trace(
                    go.Scatter(
                        x=df["Datetime"],
                        y=df[col_name],
                        name=display_name,
                        
                        # --- SAFE LEGEND GROUPING ---
                        legendgroup=str(row_idx),
                        legendgrouptitle_text=f"<b>{panel['title']}</b>",
                        # ----------------------------
                        
                        line=dict(
                            color=line_color,
                            dash=line_dash,
                            width=line_width
                        )
                    ),
                    row=row_idx,
                    col=1,
                    secondary_y=is_secondary
                )

        # --- 2. Draw Expected Range ---
        if "expected_range" in panel:
            ymin, ymax = panel["expected_range"]
            range_label = panel.get("expected_label", "Expected Range")
            range_color = panel.get("range_color", "rgba(46, 204, 113, 0.15)") 
            
            fig.add_hrect(
                y0=ymin, y1=ymax,
                fillcolor=range_color,
                line_width=0, layer="below",
                row=row_idx, col=1,
                secondary_y=False
            )
            
            first_valid_time = df["Datetime"].iloc[0] if not df.empty else None
            fig.add_trace(
                go.Scatter(
                    x=[first_valid_time], y=[None],
                    mode="markers",
                    marker=dict(size=10, color=range_color, symbol="square"),
                    name=f"{range_label} ({ymin}-{ymax})",
                    
                    # --- SAFE LEGEND GROUPING ---
                    legendgroup=str(row_idx),
                    showlegend=True
                    # ----------------------------
                ),
                row=row_idx, col=1,
                secondary_y=False
            )
                
        # --- 3. Set Primary Y-axis Title and Range ---
        primary_y_kwargs = {"title_text": panel.get("y_label", "")}
        if "y_range" in panel:
            primary_y_kwargs["range"] = panel["y_range"]
        fig.update_yaxes(**primary_y_kwargs, row=row_idx, col=1, secondary_y=False)
        
        # --- 4. Set Secondary Y-axis Title and Range ---
        if "secondary_y_label" in panel or "secondary_y_range" in panel:
            secondary_y_kwargs = {}
            if "secondary_y_label" in panel:
                secondary_y_kwargs["title_text"] = panel["secondary_y_label"]
            if "secondary_y_range" in panel:
                secondary_y_kwargs["range"] = panel["secondary_y_range"]
            fig.update_yaxes(**secondary_y_kwargs, row=row_idx, col=1, secondary_y=True)
            
    # 5. Global Layout
    fig.update_layout(
        height=max(400, row_height * total_rows),
        title_text=f"{zone_name} Dashboard",
        hovermode="x unified",
        
        # --- Single, cleanly grouped legend on the right ---
        showlegend=True,
        legend=dict(
            groupclick="toggleitem", # Allows hiding individual lines instead of the whole group
            tracegroupgap=15         # Adds nice spacing between subplot groups
        ),
        margin=dict(r=150)
    )
    
    return fig

# EKF MONITOR

In [11]:
# Loop from 1 to 5 to generate EKF diagnostic plots for all zones
for i in range(1, 6):
    zone = f"SPACE{i}-1" # 'zone' is defined here first!
    
    # ekf_config is defined INSIDE the loop so the f-strings can use the current 'zone'
    ekf_config = [
        {
            "title": f"EKF Innovations (Residuals) - Temp & CO2",
            "y_label": "Temperature Error (°C)",
            "secondary_y_label": "CO2 Error (ppm)",
            "expected_range": [-0.5, 0.5], 
            "range_color": "rgba(46, 204, 113, 0.1)",
            "expected_label": "Zero-Mean Band (Temp)",
            "traces": [
                # "Zone" gives a solid blue line for primary residual
                {"col": f"{zone}_EKF_y_T_in", "source": "Zone", "name": "Temp Residual (y_T)"},
                # "Other_1" gives a solid orange line for the secondary axis
                {"col": f"{zone}_EKF_y_C_in", "source": "Other_1", "name": "CO2 Residual (y_C)", "secondary_y": True},
            ]
        },
        {
            "title": f"EKF Innovations (Residuals) - Humidity",
            "y_label": "Humidity Error (kg/kg)",
            "expected_range": [-0.0005, 0.0005], 
            "range_color": "rgba(46, 204, 113, 0.1)",
            "expected_label": "Zero-Mean Band",
            "traces": [
                {"col": f"{zone}_EKF_y_W_in", "source": "Zone", "name": "Humidity Residual (y_W)"},
            ]
        },
        {
            "title": f"Normalized Innovation Squared (NIS)",
            "y_label": "NIS Value",
            "y_range": [0, 15],
            "expected_range": [0.216, 7.815], 
            "range_color": "rgba(241, 196, 15, 0.1)",
            "expected_label": "95% Confidence Interval (m=3)",
            "traces": [
                {"col": f"{zone}_EKF_NIS", "source": "Zone", "name": "NIS (ε)"},
            ]
        },
        {
            "title": f"Estimated Disturbances (Random Walks)",
            "y_label": "Temp Disturbance (d_T)",
            "secondary_y_label": "Hum Disturbance (d_W)",
            "traces": [
                {"col": f"{zone}_EKF_x_d_T", "source": "Zone", "name": "Estimated d_T"},
                {"col": f"{zone}_EKF_x_d_W", "source": "Other_1", "name": "Estimated d_W", "secondary_y": True},
            ]
        },
        {
            "title": f"Occupancy Estimation Tracking",
            "y_label": "Number of Occupants",
            "traces": [
                # "Setpoint" gives a thin dotted line for the actual ground truth
                {"col": f"{zone}_Occupants",     "source": "Setpoint", "name": "Actual Occupancy"},
                # "Zone" gives a solid blue line for the EKF's estimate
                {"col": f"{zone}_EKF_x_N_occ",   "source": "Zone",     "name": "Estimated Occupancy"},
            ]
        },
        {
            "title": f"Covariance Convergence (Trace of P)",
            "y_label": "Trace(P)",
            "traces": [
                {"col": f"{zone}_EKF_P_trace", "source": "Zone",    "name": "Total Uncertainty Trace"},
                {"col": f"{zone}_EKF_P_T_in",  "source": "Outside", "name": "Temp Variance P(1,1)"},
            ]
        },
        {
            "title": f"Thermal Mass & Structural Parameters",
            "y_label": "Conductance (α)",
            "secondary_y_label": "Thermal Mass Temp (°C)",
            "traces": [
                {"col": f"{zone}_EKF_x_alpha_int", "source": "Zone",    "name": "Internal Conductance (α_int)"},
                {"col": f"{zone}_EKF_x_alpha_ext", "source": "Supply",  "name": "External Conductance (α_ext)"},
                {"col": f"{zone}_EKF_x_T_m",       "source": "Other_1", "name": "Estimated Wall Temp (T_m)", "secondary_y": True},
            ]
        }
    ]

    fig = build_zone_subplots(df, zone, ekf_config)
    fig.update_layout(height=1800) 
    fig.show()